<a href="https://colab.research.google.com/github/HereLiesAz/CueDetat/blob/main/ml/pocket_detector_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cue d'Etat — Pocket Detector Training

Trains a **YOLOv8n** model on a billiards dataset and exports **TFLite FP16**
ready to drop into the `PocketDetector` interface in the app.

**Runtime:** GPU — Runtime → Change runtime type → T4 GPU.

The dataset may include multiple classes (balls, table, pockets, etc.).
The model is trained on all classes as-is. You only need to know which
class index corresponds to pockets — set that in Section 2 and the
smoke-test and Android integration will use it correctly.

## 1. Install

In [2]:
!pip install -q ultralytics roboflow
import ultralytics
ultralytics.checks()

Ultralytics 8.4.26 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 21.5/107.7 GB disk)


In [3]:
import os
from google.colab import drive
from roboflow import Roboflow

# 1. Ensure Drive is mounted
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# 2. Define and create the target directory on Drive
dataset_path = '/content/drive/MyDrive/billiards_training/datasets'
os.makedirs(dataset_path, exist_ok=True)

# 3. Change directory to save the download there
%cd {dataset_path}

# 4. Download the dataset
rf = Roboflow(api_key="sPwF0dxJDJId1E7SFQFp")
project = rf.workspace("leonardo-wijaya-bdcih").project("pool-table-detection")
version = project.version(6)
dataset = version.download("yolov8")

print(f"Dataset downloaded to: {os.getcwd()}")

/content/drive/MyDrive/billiards_training/datasets
loading Roboflow workspace...
loading Roboflow project...
Dataset downloaded to: /content/drive/MyDrive/billiards_training/datasets


In [4]:
import zipfile
import os
import yaml

# List of zip files to extract
datasets_zips = [
    '/content/CueTor Billiards Training.v8i.yolov8.zip',
    '/content/Pool Table Detection.v6i.yolov8.zip'
]

def extract_and_inspect(zip_path):
    folder_name = os.path.basename(zip_path).replace('.zip', '').replace(' ', '_')
    extract_to = f'/content/{folder_name}'

    print(f'--- Processing: {os.path.basename(zip_path)} ---')
    try:
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_to)

        yaml_path = os.path.join(extract_to, 'data.yaml')
        if os.path.exists(yaml_path):
            with open(yaml_path, 'r') as f:
                data = yaml.safe_load(f)
                names = data.get('names', [])
                print(f'Location: {extract_to}')
                print(f'Classes: {names}')
        return extract_to
    except Exception as e:
        print(f'Error extracting {zip_path}: {e}')
        return None

# Extract zip files and keep track of folders
extracted_paths = []
for ds in datasets_zips:
    if os.path.exists(ds):
        path = extract_and_inspect(ds)
        if path: extracted_paths.append(path)
    else:
        print(f'File not found: {ds}')

# Add the Roboflow downloaded folder to the list
roboflow_dir = '/content/pool-balls-yolov11m-14'
if os.path.exists(roboflow_dir):
    extracted_paths.append(roboflow_dir)
    print(f'Added Roboflow dataset: {roboflow_dir}')

File not found: /content/CueTor Billiards Training.v8i.yolov8.zip
File not found: /content/Pool Table Detection.v6i.yolov8.zip


In [5]:
import os

# Manually re-populate extracted_paths based on existing directories
extracted_paths = []

# Check for the Roboflow folder
roboflow_dir = '/content/pool-balls-yolov11m-14'
if os.path.exists(roboflow_dir):
    extracted_paths.append(roboflow_dir)
    print(f"Found Roboflow dataset: {roboflow_dir}")

# Check for any other folders created by extraction (excluding standard Colab folders)
for item in os.listdir('/content'):
    full_path = os.path.join('/content', item)
    if os.path.isdir(full_path) and item not in ['.config', 'drive', 'sample_data', 'combined_dataset']:
        if full_path not in extracted_paths:
            extracted_paths.append(full_path)
            print(f"Found extracted dataset: {full_path}")

print(f"Final list of datasets to merge: {extracted_paths}")

Final list of datasets to merge: []


In [6]:
import glob
import os

# Search for any zip files in the content directory
found_zips = glob.glob('/content/*.zip')
print(f'Found zip files: {found_zips}')

# Filter for relevant datasets and update the list for cell 5761cd6d
datasets_zips = [f for f in found_zips if 'Billiards' in f or 'Pool' in f]

if not datasets_zips:
    print('No matching zip files found. Please upload your dataset zips to the /content folder.')
else:
    print(f'Updated dataset list to: {datasets_zips}')

# Re-run the inspection/extraction logic if paths are found
for ds in datasets_zips:
    path = extract_and_inspect(ds)
    if path and path not in extracted_paths:
        extracted_paths.append(path)

Found zip files: []
No matching zip files found. Please upload your dataset zips to the /content folder.


In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
!pip install -q roboflow
import os
from roboflow import Roboflow
from google.colab import drive

if not os.path.exists('/content/drive'): drive.mount('/content/drive')

# Target path on Drive
balls_path = '/content/drive/MyDrive/billiards_training/datasets/balls_dataset'
os.makedirs(balls_path, exist_ok=True)
%cd {balls_path}

rf = Roboflow(api_key="sPwF0dxJDJId1E7SFQFp")
project = rf.workspace("wrckmn1").project("pool-balls-yolov11m")
version = project.version(14)
dataset = version.download("yolov8")

print(f"Balls dataset downloaded to Drive: {balls_path}")

/content/drive/MyDrive/billiards_training/datasets/balls_dataset
loading Roboflow workspace...
loading Roboflow project...
Balls dataset downloaded to Drive: /content/drive/MyDrive/billiards_training/datasets/balls_dataset


In [15]:
import kagglehub
path = kagglehub.dataset_download("hereliesaz/cue-detat")

100%|██████████| 1.52G/1.52G [00:20<00:00, 78.9MB/s]

Extracting files...


In [16]:
# Find the index for 'pool-table-hole'
try:
    pocket_idx = master_classes.index('pool-table-hole')
    print(f"The index for 'pool-table-hole' is: {pocket_idx}")
    print(f"Suggested update: POCKET_CLASS_IDX = [{pocket_idx}]")
except ValueError:
    print(" 'pool-table-hole' not found in master_classes. Please check the list: ", master_classes)

NameError: name 'master_classes' is not defined

In [12]:
import glob
import os

# Search for any 'best.pt' files in the training directory
search_path = "/content/drive/MyDrive/billiards_training/**/best.pt"
found_weights = glob.glob(search_path, recursive=True)

if found_weights:
    print("Found weights at:")
    for w in sorted(found_weights):
        print(f"- {w}")

    # Pick the most recent one (usually the one with the highest number or latest timestamp)
    latest_weights = sorted(found_weights)[-1]
    print(f"\nRecommended weights for export: {latest_weights}")
else:
    print("No 'best.pt' files found. Please ensure training completed successfully.")

Found weights at:
- /content/drive/MyDrive/billiards_training/pocket_detector4/weights/best.pt

Recommended weights for export: /content/drive/MyDrive/billiards_training/pocket_detector4/weights/best.pt


In [ ]:
import os
from roboflow import Roboflow
from google.colab import drive

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

    # Target path on Drive
    cuetor_path = '/content/drive/MyDrive/billiards_training/datasets/CueTor-Billiards-Training-8'
    os.makedirs(cuetor_path, exist_ok=True)
    %cd {cuetor_path}

    # Use Universe API approach for public datasets
    rf = Roboflow(api_key="sPwF0dxJDJId1E7SFQFp")
    project = rf.universe("cuetor").project("cuetor-billiards-training")
    version = project.version(8)
    dataset = version.download("yolov8")

    print(f"CueTor dataset downloaded to Drive: {cuetor_path}")

In [17]:
import os
import shutil
import yaml

# Source directories including the new Kaggle dataset
extracted_paths = [
    '/content/drive/MyDrive/billiards_training/datasets/balls_dataset/pool-balls-yolov11m-14',
    '/content/drive/MyDrive/billiards_training/datasets/Pool-Table-Detection-6',
    '/content/drive/MyDrive/billiards_training/datasets/CueTor-Billiards-Training-8',
    path  # This variable comes from cell 42_zzxaKWr8S (Kaggle dataset)
]

combined_dir = '/content/drive/MyDrive/billiards_training/datasets/combined_dataset'
if os.path.exists(combined_dir): shutil.rmtree(combined_dir)
os.makedirs(combined_dir, exist_ok=True)

for split in ['train', 'valid', 'test']:
    os.makedirs(os.path.join(combined_dir, split, 'images'), exist_ok=True)
    os.makedirs(os.path.join(combined_dir, split, 'labels'), exist_ok=True)

master_classes = []

print("Merging ALL datasets (Drive + Kaggle)...")
for folder in extracted_paths:
    if not os.path.exists(folder):
        print(f"Warning: {folder} not found.")
        continue

    # Handle potential nested structure in Kaggle/Roboflow downloads
    y_path = os.path.join(folder, 'data.yaml')
    if not os.path.exists(y_path):
        # Check one level deeper just in case
        subdirs = [d for d in os.listdir(folder) if os.path.isdir(os.path.join(folder, d))]
        for sd in subdirs:
            if os.path.exists(os.path.join(folder, sd, 'data.yaml')):
                folder = os.path.join(folder, sd)
                y_path = os.path.join(folder, 'data.yaml')
                break

    if not os.path.exists(y_path): continue

    with open(y_path, 'r') as f:
        cfg = yaml.safe_load(f)
        names = cfg.get('names', [])
        if isinstance(names, dict): names = [names[i] for i in sorted(names.keys())]

        class_map = {}
        for i, name in enumerate(names):
            if name not in master_classes: master_classes.append(name)
            class_map[i] = master_classes.index(name)

    for split in ['train', 'valid', 'test']:
        # Some datasets use 'val' instead of 'valid'
        src_split = split if os.path.exists(os.path.join(folder, split)) else 'val' if split == 'valid' else split
        s_img, s_lbl = os.path.join(folder, src_split, 'images'), os.path.join(folder, src_split, 'labels')

        if os.path.exists(s_img):
            for f in os.listdir(s_img):
                shutil.copy2(os.path.join(s_img, f), os.path.join(combined_dir, split, 'images', f))
        if os.path.exists(s_lbl):
            for f in os.listdir(s_lbl):
                with open(os.path.join(s_lbl, f), 'r') as lf: lines = lf.readlines()
                with open(os.path.join(combined_dir, split, 'labels', f), 'w') as lf:
                    for l in lines:
                        p = l.split()
                        if p:
                            p[0] = str(class_map.get(int(p[0]), p[0]))
                            lf.write(' '.join(p) + '\n')

with open(os.path.join(combined_dir, 'data.yaml'), 'w') as f:
    yaml.dump({'train': f'{combined_dir}/train/images', 'val': f'{combined_dir}/valid/images', 'test': f'{combined_dir}/test/images', 'nc': len(master_classes), 'names': master_classes}, f)

print(f"All datasets combined on Drive: {combined_dir}")
print(f"Final Classes: {master_classes}")

Merging ALL datasets (Drive + Kaggle)...
All datasets combined on Drive: /content/drive/MyDrive/billiards_training/datasets/combined_dataset
Final Classes: ['pool-table', 'pool-table-hole', 'pool-table-side', '0', '1', '10', '11', '12', '13', '14', '15', '2', '3', '4', '5', '6', '7', '8', '9']


In [18]:
# Find the index for 'pool-table-hole'
try:
    pocket_idx = master_classes.index('pool-table-hole')
    print(f"The index for 'pool-table-hole' is: {pocket_idx}")
    print(f"Suggested update: POCKET_CLASS_IDX = [{pocket_idx}]")
except ValueError:
    print(" 'pool-table-hole' not found in master_classes. Please check the list: ", master_classes)

The index for 'pool-table-hole' is: 1
Suggested update: POCKET_CLASS_IDX = [1]


In [19]:
print(f'Total classes: {len(master_classes)}')
for i, name in enumerate(master_classes):
    print(f'Index {i}: {name}')

Total classes: 19
Index 0: pool-table
Index 1: pool-table-hole
Index 2: pool-table-side
Index 3: 0
Index 4: 1
Index 5: 10
Index 6: 11
Index 7: 12
Index 8: 13
Index 9: 14
Index 10: 15
Index 11: 2
Index 12: 3
Index 13: 4
Index 14: 5
Index 15: 6
Index 16: 7
Index 17: 8
Index 18: 9


In [ ]:
# Find the index for 'pool-table-hole'
try:
    pocket_idx = master_classes.index('pool-table-hole')
    print(f"The index for 'pool-table-hole' is: {pocket_idx}")
    print(f"Suggested update: POCKET_CLASS_IDX = [{pocket_idx}]")
except ValueError:
    print(" 'pool-table-hole' not found in master_classes. Please check the list: ", master_classes)

In [ ]:
print(f'Total classes: {len(master_classes)}')
for i, name in enumerate(master_classes):
    print(f'Index {i}: {name}')

## 2. Configuration

Fill these in, then run all cells.

## 3. Load Dataset & Inspect Classes

## 4. Train

In [ ]:
from ultralytics import YOLO
import os
import torch

# Configuration variables
EPOCHS = 50
IMGSZ = 640
BATCH = 32
POCKET_CLASS_IDX = [1]
DATASET_YAML = "/content/drive/MyDrive/billiards_training/datasets/combined_dataset/data.yaml"
PROJECT_DIR = "/content/drive/MyDrive/billiards_training"

# Auto-detect device: use GPU (0) if available, otherwise CPU
device = 0 if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load a pretrained YOLOv8n model
model = YOLO("yolov8n.pt")

# Train the model
results = model.train(
    data=DATASET_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=device,
    patience=20,
    name="pocket_detector",
    project=PROJECT_DIR,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=45.0, flipud=0.5, fliplr=0.5,
    mosaic=1.0, mixup=0.1,
    save=True,
    save_period=5
)

print(f"Training complete. Weights are in: {results.save_dir}")

Using device: cpu
Ultralytics 8.4.26 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/billiards_training/datasets/combined_dataset/data.yaml, degrees=45.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=pocket_detector5, nbs=64, nms=False, opset=None

In [ ]:
from ultralytics import YOLO
import os

# Configuration variables
EPOCHS = 50
IMGSZ = 640
BATCH = 32
DATASET_YAML = "/content/combined_dataset/data.yaml"
PROJECT_DIR = "/content/drive/MyDrive/billiards_training"

# Ensure the base directory exists before training starts
os.makedirs(PROJECT_DIR, exist_ok=True)

# Resume Logic: Check if 'last.pt' exists in the weights folder
checkpoint_path = os.path.join(PROJECT_DIR, "pocket_detector", "weights", "last.pt")
resume_training = os.path.exists(checkpoint_path)

if resume_training:
    print(f"Found checkpoint at {checkpoint_path}. Resuming training...")
    model = YOLO(checkpoint_path)
else:
    print("No checkpoint found. Starting fresh training with yolov8n.pt")
    model = YOLO("yolov8n.pt")

# Train/Resume the model
results = model.train(
    data=DATASET_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=0,
    patience=20,
    name="pocket_detector",
    project=PROJECT_DIR,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=45.0, flipud=0.5, fliplr=0.5,
    mosaic=1.0, mixup=0.1,
    save=True,
    save_period=5,
    resume=resume_training
)

print(f"Training complete or paused. Weights are in: {results.save_dir}")

In [ ]:
import os

# --- GitHub Configuration ---
# IMPORTANT: Fill these in with your actual details!
GITHUB_TOKEN = "your_token_here"
GITHUB_USER = "your_username"
GITHUB_REPO = "your_repo_name"
PROJECT_DIR = "/content/drive/MyDrive/billiards_training"

# Navigate to your drive project folder
if os.path.exists(PROJECT_DIR):
    %cd {PROJECT_DIR}

    # Initialize git if not already done
    if not os.path.exists('.git'):
        !git init
        !git remote add origin https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{GITHUB_REPO}.git
        !git branch -M main

    # Configure user
    !git config --global user.email "you@example.com"
    !git config --global user.name "{GITHUB_USER}"

    # Add, commit and push
    !git add .
    !git commit -m "Update training weights and logs"
    !git push -u origin main
    print("Changes pushed to GitHub.")
else:
    print(f"Error: {PROJECT_DIR} not found. Ensure training has started and created this directory.")

## 5. Validate

In [ ]:
import os
best_weights = os.path.join(results.save_dir, "weights", "best.pt")
trained_model = YOLO(best_weights)
metrics = trained_model.val(data=DATASET_YAML)
print(f"mAP50:     {metrics.box.map50:.4f}")
print(f"mAP50-95:  {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")
print()
if metrics.box.map50 > 0.85:
    print("✓ mAP50 > 0.85 — good to export")
else:
    print("⚠ mAP50 below 0.85 — consider more epochs or more data")

## 6. Export to TFLite FP16

In [13]:
from ultralytics import YOLO
import os

# Using the weights we just found on Drive
latest_weights = '/content/drive/MyDrive/billiards_training/pocket_detector4/weights/best.pt'
model = YOLO(latest_weights)

# Export to TFLite FP16
export_path = model.export(
    format='tflite',
    imgsz=640,
    half=True,
    int8=False,
    nms=True
)

print(f'\nExport complete! TFLite file is at: {export_path}')

Ultralytics 8.4.26 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
Model summary (fused): 73 layers, 3,006,233 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/billiards_training/pocket_detector4/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (6.0 MB)
requirements: Ultralytics requirements ['sng4onnx>=1.0.1', 'onnx_graphsurgeon>=0.3.26', 'ai-edge-litert>=1.2.0', 'onnx>=1.12.0,<2.0.0', 'onnx2tf>=1.26.3,<1.29.0', 'onnxslim>=0.1.71', 'onnxruntime'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 18 packages in 3.58s
Prepared 9 packages in 3.40s
Installed 9 packages in 678ms
 + ai-edge-litert==2.1.3
 + backports-strenum==1.3.1
 + colorama==0.4.6
 + onnx==1.20.1
 + onnx-graphsurgeon==0.5.8
 + onnx2tf==1.28.8
 + onnxruntim

## 7. Smoke-test

Shows detections for the pocket class only (filtered by `POCKET_CLASS_IDX`).

In [ ]:
import numpy as np, cv2, glob, matplotlib.pyplot as plt, tensorflow as tf

interp = tf.lite.Interpreter(model_path=export_path)
interp.allocate_tensors()
inp_d = interp.get_input_details()
out_d = interp.get_output_details()
SZ    = int(inp_d[0]['shape'][1])
dtype = inp_d[0]['dtype']
print("Input:", inp_d[0]['shape'], dtype)

def infer(path):
    bgr = cv2.imread(path)
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    h, w = rgb.shape[:2]
    inp = np.expand_dims((cv2.resize(rgb, (SZ, SZ)) / 255.0).astype(dtype), 0)
    interp.set_tensor(inp_d[0]['index'], inp)
    interp.invoke()
    return rgb, interp.get_tensor(out_d[0]['index'])[0], w, h

yaml_dir = os.path.dirname(DATASET_YAML) + "/"
imgs = (glob.glob(yaml_dir + 'valid/images/*.jpg') +
        glob.glob(yaml_dir + 'valid/images/*.png'))[:4]

fig, axes = plt.subplots(1, max(len(imgs), 1), figsize=(5 * max(len(imgs), 1), 5))
if len(imgs) == 1: axes = [axes]

for ax, p in zip(axes, imgs):
    img, dets, w, h = infer(p)
    draw, n = img.copy(), 0
    for d in dets:
        # d = [x1, y1, x2, y2, conf, class_idx]  (with baked NMS)
        conf = float(d[4]) if len(d) >= 5 else 0
        cls  = int(d[5])   if len(d) >= 6 else 0
        if conf >= CONF_THRESHOLD and cls in POCKET_CLASS_IDX:
            x1,y1,x2,y2 = (int(d[0]*w/SZ), int(d[1]*h/SZ),
                            int(d[2]*w/SZ), int(d[3]*h/SZ))
            cv2.rectangle(draw, (x1,y1), (x2,y2), (0,255,0), 2)
            cv2.putText(draw, f"{conf:.2f}", (x1, y1-5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 1)
            n += 1
    ax.imshow(draw); ax.set_title(f"{n} pockets"); ax.axis('off')

plt.tight_layout(); plt.show()

## 8. Download

In [14]:
import shutil
from google.colab import files
import os

# Ensure we point to the newly exported file
if 'export_path' in locals():
    out = '/content/pocket_detector.tflite'
    shutil.copy(export_path, out)
    print(f'Downloading: {out} ({os.path.getsize(out)/1024/1024:.1f} MB)')
    files.download(out)
else:
    print('Error: Please run the export cell (Mi8ZHDKmZ3fz) first!')

Downloading: /content/pocket_detector.tflite (6.0 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 9. Android Integration (v1.4)

1. Copy `pocket_detector.tflite` → `app/src/main/assets/`

2. `app/build.gradle.kts`:
```kotlin
implementation("org.tensorflow:tensorflow-lite:2.16.1")
implementation("org.tensorflow:tensorflow-lite-support:0.4.4")
```

3. Implement `PocketDetector` — filter output by the pocket class index from this notebook:
```kotlin
class TflitePocketDetector(context: Context) : PocketDetector {
    // Class indices that correspond to pockets in your dataset
    private val pocketClassIndices = setOf(0)  // match POCKET_CLASS_IDX above

    private val interpreter = Interpreter(
        FileUtil.loadMappedFile(context, "pocket_detector.tflite")
    )

    override fun detect(yBytes: ByteArray, width: Int, height: Int): List<PointF>? {
        // 1. Resize yBytes to 640x640, replicate Y channel to RGB
        // 2. Run interpreter
        // 3. Parse output [x1,y1,x2,y2,conf,class_idx]
        //    return centres where conf >= 0.4 && class_idx in pocketClassIndices
    }
}
```

4. Pass to `TableScanAnalyzer` in `ProtractorScreen.kt`:
```kotlin
val detector = remember { TflitePocketDetector(context) }
val tableScanAnalyzer = remember(tableScanViewModel) {
    TableScanAnalyzer(
        tableScanViewModel::onFrame,
        tableScanViewModel::onFeltColorSampled,
        pocketDetector = detector,
    )
}
```